# Bus Halt Passenger Queue Simulation

## 1. Project title and introduction

This project compares passenger waiting and service performance at a bus halt with one and two boarding bays. The one-bay case is represented by an M/M/1 queue and the two-bay case by an M/M/2 queue.

A discrete-event simulation will advance the model using passenger arrivals and boarding departures. NumPy provides random sampling, Pandas organises experiment records, Matplotlib creates graphs, and SciPy can support statistical calculations. The current notebook contains the academic structure and implementation framework; no simulation results are fabricated.

In [1]:
# Google Colab setup; uncomment when dependencies are not already available.
# !pip install numpy pandas matplotlib scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Reproducible seed:", SEED)

NumPy: 2.5.5
Pandas: 3.1.1
Reproducible seed: 42


## 2. Problem formulation

Passengers arrive independently at a bus halt and join a queue when all boarding bays are occupied. The model needs to determine how a change from one to two bays affects passenger waiting and queue-building behaviour.

The problem is formulated as a comparison of two stochastic queue configurations under controlled arrival and service rates:

## 3. Objectives

The objectives are to:

1. Represent passenger arrivals using a Poisson process.
2. Represent boarding service times using exponential distributions.
3. Develop reproducible M/M/1 and M/M/2 discrete-event simulations.
4. Measure waiting time, queue length, utilisation, throughput, and system time.
5. Compare the simulation configurations across controlled experiments.
6. Verify key implementation behaviour and provide uncertainty estimates.
7. Explain the limitations of the assumptions and interpret the final results.

## 4. Queueing theory background

An M/M/c queue has $c$ independent identical servers. Passengers arrive according to a Poisson process at rate $\lambda$, and each boarding bay completes service according to an exponential distribution with rate $\mu$.

For M/M/1, utilisation is

$$\rho_1 = \frac{\lambda}{\mu}.$$ 

For M/M/2, utilisation is

$$\rho_2 = \frac{\lambda}{2\mu},$$

where the two-bay system is stable when $\lambda < 2\mu$. Erlang C gives the probability that an arriving passenger must wait when all $c$ bays are busy.

## 5. Model assumptions

The model structure will state and justify the following assumptions:

- Passenger arrivals follow a stationary Poisson process.
- All arrival and service observations are independent.
- Boarding service times are exponential and independent of arrival times.
- Passenger movement follows a first-in, first-out queue discipline.
- Each boarding bay serves one passenger at a time.
- The model has no deliberate balking, reneging, or priority rules unless added and analysed separately.
- A warm-up period may be excluded when estimating steady-state performance.

These assumptions are simplified. Their effect on real bus-halt behaviour will be discussed in the conclusions.

## 6. Mathematical equations

Let inter-arrival time $A$, arrival rate $\lambda$, service time $S$, and service rate $\mu$. The exponential distributions are

$$A \sim \operatorname{Exp}(\lambda), \qquad f_A(a)=\lambda e^{-\lambda a},$$

$$S \sim \operatorname{Exp}(\mu), \qquad f_S(s)=\mu e^{-\mu s}.$$ 

For a passenger $i$, total time in the system is

$$W_i^{system}=W_i^{queue}+S_i,$$

where $W_i^{queue}$ is waiting before boarding. For M/M/1, steady-state mean queue length and mean waiting time are represented analytically later as

$$L_q=\frac{\rho^2}{1-\rho}, \qquad W_q=\frac{L_q}{\lambda}.$$

For M/M/2, Erlang C provides the steady-state probability that an arriving passenger waits. The exact formulas and implementation will be added only during the simulation stage.

## 7. Simulation parameters

Parameters will be stored in a small Python dictionary or structured experiment table. Suggested parameters are described below; numerical values should remain editable and should not be interpreted as study results.

In [2]:
# Parameter framework. Values are configuration placeholders for later implementation.
simulation_parameters = {
    "arrival_rate_per_hour": None,
    "service_rate_per_hour": None,
    "simulation_hours": None,
    "warm_up_hours": None,
    "replications": None,
    "random_seed": SEED,
}

pd.DataFrame([simulation_parameters])

,arrival_rate_per_hour,service_rate_per_hour,simulation_hours,warm_up_hours,replications,random_seed
0,None,None,None,None,None,42


## 8. Random number generation

All stochastic samples will use a NumPy `Generator` created with a fixed seed. Using a named seed makes reruns reproducible and allows separate scenarios to receive controlled independent streams.

This section is reserved for seed management and random-stream design. It will not record generated passenger data as model results.

In [3]:
def create_random_stream(seed: int) -> np.random.Generator:
    """Create an independent reproducible NumPy random stream."""
    return np.random.default_rng(seed)

def create_seed_list(number_of_replications: int, base_seed: int = SEED):
    """Create one seed for each planned replication."""
    return list(np.arange(base_seed, base_seed + number_of_replications, dtype=int))

create_seed_list(number_of_replications=5)

[np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]

## 9. Passenger arrival generation

Passenger arrivals can be generated from exponential inter-arrival times:

$$T_{k+1}=T_k+A_k, \qquad A_k \sim \operatorname{Exp}(\lambda).$$

For inverse transform sampling, draw $U\sim U(0,1)$ and generate each inter-arrival time as

$$A_k=-\frac{\ln(U_k)}{\lambda}, \qquad T_{k+1}=T_k+A_k.$$

The function below is reproducible with a NumPy integer seed. It stops before or at the specified simulation duration and does not generate timestamps beyond that horizon.

In [4]:
def generate_arrival_times(arrival_rate, duration, seed=42):
    """Generate arrival timestamps using inverse transform sampling."""
    if isinstance(arrival_rate, bool) or not isinstance(arrival_rate, (int, float)):
        raise TypeError("arrival_rate must be a real number")
    if isinstance(duration, bool) or not isinstance(duration, (int, float)):
        raise TypeError("duration must be a real number")

    arrival_rate = float(arrival_rate)
    duration = float(duration)

    if not np.isfinite(arrival_rate) or arrival_rate <= 0:
        raise ValueError("arrival_rate must be a finite positive number")
    if not np.isfinite(duration) or duration < 0:
        raise ValueError("duration must be a finite non-negative number")

    if isinstance(seed, bool) or (seed is not None and not isinstance(seed, (int, np.random.Generator))):
        raise TypeError("seed must be an integer, a NumPy Generator, or None")
    if isinstance(seed, int) and seed < 0:
        raise ValueError("seed must be non-negative")

    rng = seed if isinstance(seed, np.random.Generator) else np.random.default_rng(seed)

    if duration == 0:
        return np.empty(0, dtype=float)

    timestamps = []
    elapsed = 0.0

    while True:
        uniforms = np.maximum(rng.random(1024), np.finfo(float).eps)
        inter_arrivals = -np.log(uniforms) / arrival_rate
        cumulative = elapsed + np.cumsum(inter_arrivals)
        timestamps.extend(cumulative[cumulative <= duration].tolist())

        if cumulative[-1] > duration:
            return np.asarray(timestamps, dtype=float)

        elapsed = float(cumulative[-1])

### Example generated arrival times

The example below uses an arrival rate of $\lambda=2$ passengers per time unit for a 10-unit observation period. The same seed reproduces the timestamps.

In [5]:
example_times = generate_arrival_times(arrival_rate=2.0, duration=10.0, seed=2026)

example_arrivals = pd.DataFrame({
    "passenger": np.arange(1, len(example_times) + 1),
    "arrival_time": example_times,
})

## 10. Passenger service-time generation

For one boarding bay, passenger service duration $S$ follows an exponential distribution with rate $\mu$:

$$F_S(s)=1-e^{-\mu s}, \qquad s\geq 0.$$

Inverse transform sampling sets $U=F_S(S)$ for $U\sim U(0,1)$ and solves for $S$:

$$S=-\frac{\ln(1-U)}{\mu}.$$

Since $1-U$ is also uniform on $(0,1)$, the implementation equivalently computes $S=-\ln(U)/\mu$. Thus every duration is positive, and the expected service duration is $1/\mu$.

The arrival rate $\lambda$ is the average number of passengers arriving per time unit. The service rate $\mu$ is the average number served by one boarding bay per time unit. With two identical bays, the combined nominal capacity is $2\mu$; $\mu$ itself remains the rate for each bay.

The function below returns one service duration per passenger and accepts an integer seed or NumPy `Generator` for reproducible experiments.

In [ ]:
from numbers import Real


def generate_service_times(service_rate, passenger_count, seed=42):
    """Generate exponential boarding durations for one bay's service rate."""
    if isinstance(service_rate, bool) or not isinstance(service_rate, Real):
        raise TypeError("service_rate must be a real number")
    if isinstance(passenger_count, bool) or not isinstance(passenger_count, int):
        raise TypeError("passenger_count must be an integer")

    service_rate = float(service_rate)
    if not np.isfinite(service_rate) or service_rate <= 0:
        raise ValueError("service_rate must be a finite positive number")
    if passenger_count < 0:
        raise ValueError("passenger_count must be non-negative")

    if isinstance(seed, np.random.Generator):
        rng = seed
    elif isinstance(seed, bool) or (seed is not None and not isinstance(seed, int)):
        raise TypeError("seed must be an integer, a NumPy Generator, or None")
    elif seed is not None and seed < 0:
        raise ValueError("seed must be non-negative")
    else:
        rng = np.random.default_rng(seed)

    if passenger_count == 0:
        return np.empty(0, dtype=float)

    uniforms = np.maximum(rng.random(passenger_count), np.finfo(float).eps)
    return -np.log(uniforms) / service_rate

### Small service-time example

For example, $μ=0.5$ passengers per time unit means a mean service duration of $1/0.5=2$ time units. The array below gives the generated duration for six passengers using a fixed seed.

In [ ]:
example_service_times = generate_service_times(
    service_rate=0.5,
    passenger_count=6,
    seed=2026,
)

pd.DataFrame({
    "passenger": np.arange(1, len(example_service_times) + 1),
    "service_duration": example_service_times,
})

In [ ]:
service_check = generate_service_times(0.5, passenger_count=10000, seed=77)
repeat_check = generate_service_times(0.5, passenger_count=10000, seed=77)
different_seed_check = generate_service_times(0.5, passenger_count=10000, seed=78)
fast_service_check = generate_service_times(2.0, passenger_count=10000, seed=77)

assert np.all(np.isfinite(service_check))
assert np.all(service_check > 0)
np.testing.assert_array_equal(service_check, repeat_check)
assert not np.array_equal(service_check, different_seed_check)
assert np.mean(fast_service_check) < np.mean(service_check)
assert np.isclose(np.mean(fast_service_check), 0.5, rtol=0.05)
assert generate_service_times(1.0, passenger_count=0).size == 0

for invalid_rate in (0, -0.5, np.inf, np.nan):
    try:
        generate_service_times(invalid_rate, passenger_count=1)
    except ValueError:
        continue
    raise AssertionError(f"Expected invalid service rate to fail: {invalid_rate}")

print("Service-time checks passed.")

## 11. M/M/1 discrete-event simulation

`simulate_mm1` advances the clock to whichever event happens next: a passenger arrival or the current service completion. Between events, it accumulates the area under the waiting-queue length and the time the bay is busy. An arrival starts service immediately when the bay is idle; otherwise the passenger joins the end of the shared FIFO queue. A service completion takes precedence over an arrival at the same timestamp.

The result records each passenger's arrival, service start, scheduled completion, waiting time, and horizon status. `passengers_waiting`, `passengers_in_service`, and `completed_passengers` report the final state. A passenger still waiting has an unknown final wait (`NaN`); an in-service passenger can have a scheduled completion after the horizon, but is not counted as completed.

The reported average waiting time uses passengers who started service. Time-average queue length is the integral of the number waiting divided by the observation duration; utilisation is busy time divided by duration; throughput is completed passengers divided by duration. Finite-horizon simulation is allowed for any positive rates, while the `stable` result flag is true only when $\lambda<\mu$.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "queue_simulation.py").is_file():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from queue_simulation import simulate_mm1, theoretical_mm1_metrics

## 12. M/M/2 discrete-event simulation

`simulate_mm2` uses two independent server slots and one shared FIFO queue. At each event, it compares the next arrival with the earliest bay completion. A completion frees that specific bay, which immediately takes the oldest waiting passenger if any; an arrival uses the lowest-index available bay, making the selection deterministic when both bays are idle. If a completion and arrival tie, the completion is processed first.

Each passenger keeps one-based bay assignment plus arrival, service-start, scheduled-completion, and waiting-time records. The result reports both bay utilizations, their average as overall utilization, queue time-average, throughput, and each passenger's status at the horizon. Unfinished passengers remain waiting or in service rather than being marked complete.

For identical bays, define offered load $a=\lambda/\mu$ and per-bay utilisation $\rho=\lambda/(2\mu)$. Erlang C gives

$$P_0=\left[1+a+\frac{a^2}{2(1-\rho)}\right]^{-1}, \qquad P_{wait}=\frac{a^2}{2(1-\rho)}P_0,$$

$$W_q=\frac{P_{wait}}{2\mu-\lambda}, \qquad L_q=\lambda W_q, \qquad W=W_q+\frac{1}{\mu}, \qquad L=\lambda W.$$

Steady-state comparisons require $\lambda<2\mu$. The finite-horizon simulator can still run outside that region, but marks `stable=False`; `theoretical_mm2_metrics` rejects those unstable inputs.

In [ ]:
from queue_simulation import simulate_mm2, theoretical_mm2_metrics

In [ ]:
mm2_verification_run = simulate_mm2(
    arrival_rate=1.2,
    service_rate=1.0,
    duration=100000.0,
    seed=2027,
)
mm2_theory = theoretical_mm2_metrics(1.2, 1.0)
mm2_completed = mm2_verification_run["passenger_status"] == "completed"
mm2_system_times = (
    mm2_verification_run["service_completion_times"][mm2_completed]
    - mm2_verification_run["arrival_times"][mm2_completed]
)
mm2_simulated_metrics = {
    "rho": mm2_verification_run["overall_utilization"],
    "Wq": mm2_verification_run["average_waiting_time"],
    "Lq": mm2_verification_run["time_average_queue_length"],
    "W": float(np.mean(mm2_system_times)),
}

for metric, simulated_value in mm2_simulated_metrics.items():
    theoretical_value = mm2_theory[metric]
    print(
        f"{metric}: theory={theoretical_value:.4f}, "
        f"simulation={simulated_value:.4f}, "
        f"absolute_error={abs(simulated_value - theoretical_value):.4f}"
    )

print("Arrivals:", len(mm2_verification_run["arrival_times"]))
print("Completed:", len(mm2_verification_run["completed_passengers"]))
print("Waiting at horizon:", len(mm2_verification_run["passengers_waiting"]))
print("In service at horizon:", len(mm2_verification_run["passengers_in_service"]))
print("Per-bay utilization:", mm2_verification_run["per_server_utilization"])
print("Throughput:", mm2_verification_run["throughput"])

## 13. Performance measurement

The simulation reports average waiting time for passengers whose service has started, time-average number of passengers waiting, server utilisation, and completed-passenger throughput. Passenger-level arrays retain event timestamps and horizon status so censored passengers are not misreported as completed.

In [10]:
performance_columns = [
    "boarding_bays",
    "replication",
    "arrival_rate",
    "service_rate",
    "mean_wait",
    "mean_queue_length",
    "mean_system_time",
    "utilisation",
    "throughput",
]

pd.DataFrame(columns=performance_columns)

,boarding_bays,replication,arrival_rate,service_rate,mean_wait,mean_queue_length,mean_system_time,utilisation,throughput


## 14. Verification against M/M/1 theory

For a stable M/M/1 system ($\lambda<\mu$), `theoretical_mm1_metrics` calculates

$$\rho=\frac{\lambda}{\mu}, \qquad W_q=\frac{\lambda}{\mu(\mu-\lambda)}, \qquad L_q=\frac{\lambda^2}{\mu(\mu-\lambda)}, \qquad W=\frac{1}{\mu-\lambda}.$$

It rejects unstable parameter pairs because these steady-state quantities do not apply. The finite-horizon simulator itself still permits unstable rates and reports `stable=False`.

The run below uses $\lambda=0.5$, $\mu=1.0$, duration 100,000 time units, and seed 2026. It compares simulated utilisation with $\rho$, average waiting with $W_q$, time-average queue length with $L_q$, and completed passengers' mean system time with $W$. Finite-run differences are stochastic, not evidence that the formulas must match exactly. Incomplete passengers are reported separately at the horizon.

In [ ]:
verification_run = simulate_mm1(
    arrival_rate=0.5,
    service_rate=1.0,
    duration=100000.0,
    seed=2026,
)
verification_theory = theoretical_mm1_metrics(0.5, 1.0)
completed_mask = verification_run["passenger_status"] == "completed"
completed_system_times = (
    verification_run["service_completion_times"][completed_mask]
    - verification_run["arrival_times"][completed_mask]
)
simulated_metrics = {
    "rho": verification_run["server_utilization"],
    "Wq": verification_run["average_waiting_time"],
    "Lq": verification_run["time_average_queue_length"],
    "W": float(np.mean(completed_system_times)),
}

for metric, simulated_value in simulated_metrics.items():
    theoretical_value = verification_theory[metric]
    print(
        f"{metric}: theory={theoretical_value:.4f}, "
        f"simulation={simulated_value:.4f}, "
        f"absolute_error={abs(simulated_value - theoretical_value):.4f}"
    )

print("Arrivals:", len(verification_run["arrival_times"]))
print("Completed:", len(verification_run["completed_passengers"]))
print("Waiting at horizon:", len(verification_run["passengers_waiting"]))
print("In service at horizon:", len(verification_run["passengers_in_service"]))
print("Throughput:", verification_run["throughput"])

## 15. Experiments

The experiment stage will vary arrival rates, service rates, run durations, and replication counts. Parameters should be selected to cover stable and unstable regions only if the instability is intentional and clearly interpreted.

The one-bay and two-bay models should be run under equivalent arrival and service parameters. Replication results should be retained so that confidence intervals and sensitivity analysis can be calculated.

## 16. Graphs and analysis

After successful implementation, Matplotlib will be used to create graphs comparing the M/M/1 and M/M/2 configurations. Suitable examples include:

- mean waiting time against arrival rate;
- mean queue length against arrival rate;
- utilisation against arrival rate;
- throughput against arrival rate; and
- confidence intervals or replications across parameter values.

No graph is populated here because it would require invented or unvalidated experimental output.

## 17. Conclusions

The final conclusions will be written only after simulation runs and verification checks are complete. They should state:

- the most important observed difference between one and two bays;
- whether and when adding a second bay provides meaningful waiting-time improvement;
- agreement or disagreement with analytical queueing formulas;
- uncertainty and replication evidence;
- practical limitations caused by the Markovian assumptions; and
- possible future extensions, such as observed passenger-arrival data or time-varying arrival rates.

No conclusion or measured improvement is claimed in this notebook structure.